In [ ]:
import os

# Check files inside the directory
file_path = "../data/raw/bbmp/"
print("Files in bbmp directory:")
print(os.listdir(file_path))

In [ ]:
import os

path = "../data/raw/bbmp/citizen-grievances.csv"

print("Is it a folder?", os.path.isdir(path))
print("Is it a file?", os.path.isfile(path))

In [ ]:
import pandas as pd

# Target the actual CSV file inside the unzipped folder
df = pd.read_csv("../data/raw/bbmp/citizen-grievances.csv/citizen-grievances.csv")

print("=== SHAPE ===")
print(df.shape)

print("\n=== DATA INFO ===")
df.info()

print("\n=== FIRST 3 ROWS ===")
df.head(3)

In [ ]:
# 1. Standardize column names to lowercase with underscores
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# 2. Convert Grievance Date to proper datetime
df['grievance_date'] = pd.to_datetime(df['grievance_date'])

# 3. Convert repeated text fields to categorical dtypes to save memory
cat_cols = ['category', 'sub_category', 'zone_name', 'grievance_mode', 'grievance_status']
for col in cat_cols:
    df[col] = df[col].astype('category')

# Print clean summary
print("=== UPDATED DATA TYPES & MEMORY ===")
df.info()

In [ ]:
## easier version of above code
# Step 1: Easy column names
df.columns = [
    'complaint_id', 'category', 'sub_category', 'grievance_date',
    'zone_name', 'ward_name', 'grievance_mode', 'grievance_status',
    'staff_remarks', 'staff_name', 'staff_designation', 'rating'
]

# Step 2: Make dates real calendar dates
df['grievance_date'] = pd.to_datetime(df['grievance_date'])

# Step 3: Show missing values
print("=== MISSING VALUES PER COLUMN ===")
print(df.isnull().sum())

In [ ]:
df['rating']=df['rating'].fillna(0)

In [ ]:
df['rating'].isnull().sum()

In [ ]:
# 1. List of text columns that have missing values
text_columns = ['zone_name', 'grievance_mode', 'grievance_status', 'staff_remarks', 'staff_name', 'staff_designation']

# 2. Convert them to string and fill missing values in one line
for col in text_columns:
    df[col] = df[col].astype(str).fillna('Unspecified')

In [ ]:
df.isnull().sum()

In [ ]:
df['rating_given'] = (df['rating'] > 0).astype(int)


In [ ]:
# Fill missing values in the original 'rating' column with 0 (or a neutral score like 0.0)
df['rating'] = df['rating'].fillna(0)

# Verify that there are no remaining missing values in 'rating'
print("Missing values in 'rating':", df['rating'].isnull().sum())
df[['rating', 'rating_given']].head()

In [ ]:
# Identify all text (object) columns
text_columns = df.select_dtypes(include=['object']).columns

# Strip extra spaces and convert to lowercase
for col in text_columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

# Verify text columns
df[text_columns].head()



In [ ]:
import numpy as np

# Replace dummy placeholder strings with actual missing values (NaN)
df['grievance_mode'] = df['grievance_mode'].replace('-select-', np.nan)

# Check missing value count in grievance_mode
print("Missing values in 'grievance_mode':", df['grievance_mode'].isnull().sum())
df[['grievance_mode', 'grievance_status']].head()

In [ ]:
# Fill missing values in 'grievance_mode' with 'unknown'
df['grievance_mode'] = df['grievance_mode'].fillna('unknown')

# Verify no missing values remain in 'grievance_mode'
print("Missing values in 'grievance_mode':", df['grievance_mode'].isnull().sum())
df[['grievance_mode', 'grievance_status']].head()

In [ ]:
# List staff-related columns
staff_cols = ['staff_remarks', 'staff_name', 'staff_designation']

# Fill missing values in staff columns with 'not assigned'
df[staff_cols] = df[staff_cols].fillna('not assigned')

# Verify no missing values remain in these columns
print("Remaining missing values in staff columns:")
print(df[staff_cols].isnull().sum())
df[staff_cols].head()

In [ ]:
# Check total duplicate rows before dropping
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows found: {duplicate_count}")

# Drop exact duplicate rows
df = df.drop_duplicates()

# Confirm shape after dropping duplicates
print(f"Dataset shape after deduplication: {df.shape}")

In [ ]:
df.info()

In [ ]:
# Convert text columns to category dtype to optimize memory usage
str_cols = ['zone_name', 'ward_name', 'grievance_mode', 'grievance_status', 'staff_designation']

for col in str_cols:
    df[col] = df[col].astype('category')

# Check updated memory usage
df.info()

In [ ]:
# Save clean dataset to your data folder
df.to_csv('../data/civicsense_cleaned.csv', index=False)
print("Cleaned dataset successfully saved to '../data/civicsense_cleaned.csv'!")

In [ ]:
import os

# Get the absolute full path of your cleaned CSV file
file_path = os.path.abspath('../data/civicsense_cleaned.csv')

print("Your file is saved here:")
print(file_path)

In [ ]:
import os
import pandas as pd

# 1. Step up one level to find the data folder
input_path = "../data/civicsense_cleaned.csv"
output_path = "../data/unique_categories.csv"

# 2. Read the cleaned dataset
df = pd.read_csv(input_path)

# 3. Extract unique category-subcategory pairs
category_list = (
    df[["category", "sub_category"]]
    .drop_duplicates()
    .sort_values(["category", "sub_category"])
    .reset_index(drop=True)
)

# 4. Save unique_categories.csv into the data folder
category_list.to_csv(output_path, index=False)

print(f"Extraction complete! Found {len(category_list)} unique pairs:\n")
print(category_list.to_string())

In [ ]:
import os
import pandas as pd

# Load unique categories
df_cats = pd.read_csv("../data/unique_categories.csv")

def assign_bbmp_department(row):
    cat = str(row['category']).strip()
    sub = str(row['sub_category']).strip()
    
    # 1. Road Maintenance & PWD/Engineering Works
    if cat == 'Road Maintenance(Engg)':
        return 'BBMP Engineering (Road Maintenance)'
    if cat in ['Projects Central', 'Major Roads', 'Road Infrastructure', 'Optical Fiber Cables (OFC)']:
        return 'BBMP Engineering (Infrastructure & Projects)'
    if cat == 'Storm  Water Drain(SWD)':
        return 'BBMP Engineering (Storm Water Drains)'
    
    # 2. Solid Waste & Sanitation
    if cat in ['Solid Waste (Garbage) Related', 'Sanitation', 'Plastic']:
        return 'Solid Waste Management (SWM)'
    
    # 3. Revenue & Khata
    if cat in ['E khata / Khata services', 'Property Tax services', 'Revenue Department']:
        return 'Revenue & Taxation Department'
    
    # 4. Electrical
    if cat == 'Electrical':
        return 'BBMP Electrical Wing'
    
    # 5. Public Health & Food Security
    if cat in ['Health Dept', 'Indira Canteen']:
        return 'Public Health & Medical Services'
    
    # 6. Veterinary
    if cat == 'veterinary':
        return 'Animal Husbandry & Veterinary Services'
    
    # 7. Forestry & Parks
    if cat in ['Forest', 'Parks and Play grounds']:
        return 'Forestry & Horticulture Wing'
    
    # 8. Lakes & Water
    if cat in ['Lakes', 'Water Crisis']:
        return 'Lakes & Water Resources'
    
    # 9. Town Planning, Advertisements & Estate
    if cat in ['Town Planning', 'Estate', 'Advertisement', 'Markets']:
        return 'Town Planning & Asset Management'
    
    # 10. Social Welfare & Education
    if cat in ['Education', 'Welfare Schemes']:
        return 'Social Welfare & Education Department'
    
    # 11. Administration, IT & Traffic
    if cat in ['Information Technology', 'Call Center', 'BBMP Election Branch', 'CORONA COVID19']:
        return 'Administration & IT Cell'
    if cat == 'Traffic Engineer Cell (TEC)':
        return 'Traffic Engineering Cell (TEC)'
    
    return 'General Municipal Administration'

df_cats['department'] = df_cats.apply(assign_bbmp_department, axis=1)
df_cats['mapping_source'] = 'BBMP Civic Governance Rules'
df_cats['mapping_status'] = 'verified'

# Save updated mapping
df_cats.to_csv("../data/category_department_mapping.csv", index=False)

print("Updated category_department_mapping.csv saved successfully!")
print(df_cats[['category', 'sub_category', 'department']].head(10))

In [ ]:
import os
import pandas as pd

# Load cleaned dataset
df = pd.read_csv("../data/civicsense_cleaned.csv")

# Create analytics directory if it doesn't exist
os.makedirs("../analytics", exist_ok=True)

# 1. Category Analysis (Volume per category)
category_analysis = (
    df.groupby("category")
      .size()
      .reset_index(name="complaint_count")
      .sort_values("complaint_count", ascending=False)
)
category_analysis.to_csv("../analytics/category_analysis.csv", index=False)

# 2. Ward Analysis (Volume per zone & ward)
ward_analysis = (
    df.groupby(["zone_name", "ward_name"])
      .size()
      .reset_index(name="complaint_count")
      .sort_values("complaint_count", ascending=False)
)
ward_analysis.to_csv("../analytics/ward_analysis.csv", index=False)

# 3. Status Analysis (Breakdown of complaint resolution status)
status_analysis = (
    df["grievance_status"]
      .value_counts()
      .reset_index()
)
status_analysis.columns = ["grievance_status", "complaint_count"]
status_analysis.to_csv("../analytics/status_analysis.csv", index=False)

print("Analytics files successfully generated in 'analytics/' folder!")
print("\n--- Top 5 Categories ---")
print(category_analysis.head())
print("\n--- Top 5 Wards ---")
print(ward_analysis.head())
print("\n--- Status Breakdown ---")
print(status_analysis)

In [ ]:
import os
import pandas as pd

# 1. Load unique categories
df_cats = pd.read_csv("../data/unique_categories.csv")

def audit_bbmp_department(row):
    cat = str(row['category']).strip()
    sub = str(row['sub_category']).strip()
    
    # Default values
    dept = 'General Municipal Administration'
    status = 'verified'
    note = ''
    
    # Flagged Red-Flag / Suspicious Semantic Rows for Review
    if cat == 'Major Roads' and sub == 'Food Quality':
        dept = 'BBMP Engineering (Infrastructure & Projects)'
        status = 'review'
        note = 'Potential taxonomy anomaly: Food Quality listed under Major Roads'
    elif cat == 'Road Infrastructure' and sub == 'Illegal Parking':
        dept = 'Traffic Engineering Cell (TEC)'
        status = 'review'
        note = 'Traffic enforcement vs infrastructure assignment needs source check'
    elif cat == 'Road Maintenance(Engg)' and sub == 'Garbage vehicle not arrived':
        dept = 'BBMP Engineering (Road Maintenance)'
        status = 'review'
        note = 'Cross-department issue: Garbage vehicle appears under Road Engg'
    elif cat == 'Road Maintenance(Engg)' and sub == 'Street Light Not Working':
        dept = 'BBMP Engineering (Road Maintenance)'
        status = 'review'
        note = 'Overlaps with BBMP Electrical Wing taxonomy'
    elif cat == 'Storm  Water Drain(SWD)' and sub == 'Street Light Not Working':
        dept = 'BBMP Engineering (Storm Water Drains)'
        status = 'review'
        note = 'Semantic mismatch: Street light listed under SWD'
    elif cat == 'Sanitation' and sub == 'Dead animal(s)':
        dept = 'Solid Waste Management (SWM)'
        status = 'review'
        note = 'Overlaps with Veterinary disposal vs SWM sanitation'
    elif cat == 'Health Dept' and sub == 'Indira Canteen':
        dept = 'Public Health & Medical Services'
        status = 'review'
        note = 'Separate Indira Canteen category exists; needs taxonomy verification'
    elif cat == 'Health Dept' and sub == 'Delay in issue of Trade licence':
        dept = 'Public Health & Medical Services'
        status = 'review'
        note = 'Trade license also exists under Markets department'
    elif cat == 'Health Dept' and sub == 'Zonal Violation of the Building':
        dept = 'Public Health & Medical Services'
        status = 'review'
        note = 'Semantic mismatch: Building violation listed under Health Dept'
    elif cat == 'Water Crisis' and sub == 'Water related issue':
        dept = 'Lakes & Water Resources'
        status = 'review'
        note = 'Generic water issue; needs verification between BWSSB vs BBMP Lakes'
    
    # Standard Department Routing
    elif cat == 'Road Maintenance(Engg)':
        dept = 'BBMP Engineering (Road Maintenance)'
    elif cat in ['Projects Central', 'Major Roads', 'Road Infrastructure', 'Optical Fiber Cables (OFC)']:
        dept = 'BBMP Engineering (Infrastructure & Projects)'
    elif cat == 'Storm  Water Drain(SWD)':
        dept = 'BBMP Engineering (Storm Water Drains)'
    elif cat in ['Solid Waste (Garbage) Related', 'Sanitation', 'Plastic']:
        dept = 'Solid Waste Management (SWM)'
    elif cat in ['E khata / Khata services', 'Property Tax services', 'Revenue Department']:
        dept = 'Revenue & Taxation Department'
    elif cat == 'Electrical':
        dept = 'BBMP Electrical Wing'
    elif cat in ['Health Dept', 'Indira Canteen']:
        dept = 'Public Health & Medical Services'
    elif cat == 'veterinary':
        dept = 'Animal Husbandry & Veterinary Services'
    elif cat in ['Forest', 'Parks and Play grounds']:
        dept = 'Forestry & Horticulture Wing'
    elif cat in ['Lakes', 'Water Crisis']:
        dept = 'Lakes & Water Resources'
    elif cat in ['Town Planning', 'Estate', 'Advertisement', 'Markets']:
        dept = 'Town Planning & Asset Management'
    elif cat in ['Education', 'Welfare Schemes']:
        dept = 'Social Welfare & Education Department'
    elif cat in ['Information Technology', 'Call Center', 'BBMP Election Branch', 'CORONA COVID19']:
        dept = 'Administration & IT Cell'
    elif cat == 'Traffic Engineer Cell (TEC)':
        dept = 'Traffic Engineering Cell (TEC)'

    return pd.Series([dept, 'BBMP Official Taxonomy Rules', status, note])

# Apply updated mapping logic
df_cats[['department', 'mapping_source', 'mapping_status', 'review_notes']] = df_cats.apply(audit_bbmp_department, axis=1)

# Save audited CSV
df_cats.to_csv("../data/category_department_mapping.csv", index=False)

print("Audited mapping saved to '../data/category_department_mapping.csv'!")
print("\nMapping Status Breakdown:")
print(df_cats['mapping_status'].value_counts())

In [ ]:
import os
import pandas as pd

# Load cleaned dataset
df = pd.read_csv("../data/civicsense_cleaned.csv")

# Ensure analytics directory exists
os.makedirs("../analytics", exist_ok=True)

# Parse grievance_date safely
df["grievance_date"] = pd.to_datetime(df["grievance_date"], errors='coerce')

# Aggregate complaint count by month
monthly_trends = (
    df.assign(
        month=df["grievance_date"].dt.to_period("M").astype(str)
    )
    .groupby("month")
    .size()
    .reset_index(name="complaint_count")
    .sort_values("month")
)

# Export to analytics folder
monthly_trends.to_csv("../analytics/monthly_trends.csv", index=False)

# Verification checks requested by mentor
print("--- VERIFICATION ---")
print("Dataset rows: ", len(df))
print("Monthly total:", monthly_trends["complaint_count"].sum())

# Display preview
print("\nFirst 12 Months Preview:")
print(monthly_trends.head(12))

In [ ]:
import pandas as pd

# Load cleaned dataset
df = pd.read_csv("../data/civicsense_cleaned.csv")
df["grievance_date"] = pd.to_datetime(df["grievance_date"], errors='coerce')

print("=== STEP 1: Inspecting 1900 Records ===")
bad_dates = df[df["grievance_date"].dt.year == 1900]
print(f"1900 records total count: {len(bad_dates)}")
# Handle complaint_id dynamically if column name varies
id_col = 'complaint_id' if 'complaint_id' in df.columns else 'grievance_id' if 'grievance_id' in df.columns else df.columns[0]
print(bad_dates[[id_col, "grievance_date"]].head(50))

print("\n=== STEP 2: Checking Date Range & Yearly Counts ===")
print("Earliest date:", df["grievance_date"].min())
print("Latest date:  ", df["grievance_date"].max())
print("\nYearly Distribution:")
print(df["grievance_date"].dt.year.value_counts(dropna=False).sort_index())

print("\n=== STEP 3: Exact Value Distribution for 1900 ===")
print(bad_dates["grievance_date"].value_counts())

In [ ]:
future_dates = df[df["grievance_date"].dt.year == 2048]

print("2048 records:", len(future_dates))

print(
    future_dates[
        ["complaint_id", "grievance_date"]
    ].head(50)
)

print("\nUnique 2048 dates:")
print(future_dates["grievance_date"].value_counts().head(30))

In [ ]:
import os
print("Files found:", os.listdir("../data") if os.path.exists("../data") else os.listdir("."))

In [ ]:
print("Raw columns:", raw.columns.tolist())

In [ ]:
future = df[df["grievance_date"].dt.year == 2048].copy()

print("Number of records:", len(future))
print("Minimum complaint ID:", future["complaint_id"].min())
print("Maximum complaint ID:", future["complaint_id"].max())

future = future.sort_values("complaint_id")
id_diff = future["complaint_id"].diff().dropna()

print("\nNumber of ID gaps:", (id_diff != 1).sum())
print("Unique ID differences:")
print(id_diff.value_counts().head(10))

In [9]:
import pandas as pd

# 1. Load your raw dataset into memory
df = pd.read_csv(r"C:\Users\PRAPTHI SALIAN\Documents\important placement\civicsense\data\raw\bbmp\citizen-grievances.csv\citizen-grievances.csv")

# Print columns to verify the exact name if needed
print("Columns in raw file:", df.columns.tolist())

# 2. Use the exact raw column name ('Grievance Date') to parse and clean
df["Grievance Date"] = pd.to_datetime(df["Grievance Date"], errors="coerce")
invalid_mask = df["Grievance Date"].dt.year.isin([1900, 2048])
df.loc[invalid_mask, "Grievance Date"] = pd.NaT

# 3. Extract all time features using the correct column name
df["year"] = df["Grievance Date"].dt.year
df["month"] = df["Grievance Date"].dt.month
df["year_month"] = df["Grievance Date"].dt.to_period("M")
df["day_of_week"] = df["Grievance Date"].dt.day_name()

# 4. Save the updated full dataset
df.to_csv("../data/processed/civicsense_cleaned_v2.csv", index=False)
print("Updated dataset saved successfully!")

# 5. Group by year_month to generate monthly trends (NaT/anomalies automatically excluded)
monthly_trends = df.groupby("year_month", dropna=True).size().reset_index(name="complaint_count")
monthly_trends = monthly_trends.sort_values("year_month")

# 6. Save the cleaned monthly analysis files
monthly_trends.to_csv("../analytics/monthly_trends.csv", index=False)
monthly_trends.to_csv("../analytics/monthly_analysis_cleaned.csv", index=False)

print("Monthly trends successfully generated and saved!")
print(monthly_trends.head(12))

Columns in raw file: ['Complaint ID', 'Category', 'Sub Category', 'Grievance Date', 'Zone Name', 'Ward Name', 'Grievance Mode', 'Grievance Status', 'Staff Remarks', 'Staff Name', 'Staff Designation', 'Rating']
Updated dataset saved successfully!
Monthly trends successfully generated and saved!
   year_month  complaint_count
0     2020-02             6704
1     2020-03             9717
2     2020-04             4962
3     2020-05             7181
4     2020-06             9011
5     2020-07             7820
6     2020-08             7837
7     2020-09             9567
8     2020-10            11340
9     2020-11             9113
10    2020-12             8368
11    2021-01             8325
